<a href="https://colab.research.google.com/github/thiru-bit/SP25-690-Gugulothu/blob/main/Sentiment_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 'transformers' is for BERT, 'datasets' is to get the movie reviews
!pip install transformers datasets scikit-learn

In [ ]:
import torch # The brain of the project
import numpy as np
from datasets import load_dataset  # The library to get the movie reviews
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments  # The BERT tools
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# Download the movie reviews(loading the data)
dataset = load_dataset("imdb")
dataset

In [ ]:
# We take 2,000 reviews for training and 500 for testing/validation
# This makes sure our 'LEGO castle' is small enough to build quickly!
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(f"New training size: {len(small_train_dataset)}")
print(f"New testing size: {len(small_test_dataset)}")

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

train_data = small_train_dataset.map(tokenize_function, batched=True)
test_data = small_test_dataset.map(tokenize_function, batched=True)

In [ ]:
train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
trainer.evaluate()